# 🧬 CHITIN v2
### Causal Heuristics Isolating Transcriptomic Intervention Noise

**Objective:** Strip systematic pan-perturbation variation from metacell expression matrices, producing a pure Δ-matrix of causal perturbation signatures for downstream GRN inference.

**v2 upgrades:**
- **Auto-calibration** — Pareto sweep over (k, n_pcs, distance_metric) selects optimal params automatically
- **PC decomposition mode** — Projects out systematically-biased PCA axes instead of KNN subtraction; better for homogeneous cell lines (iPSC, hESC)
- **Hybrid mode** — PC decomposition followed by KNN on residuals
- Correction mode and all params are set in `chitin_config.yaml`

**Input:** SPORE Phase 8 metacell `.h5ad` files (train/val/test)  
**Output:** Δ-transformed `.h5ad` files  
**Downstream:** GuanLab PSGRN (LightGBM) → FUNGI → SPECTRA

---

## 1 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import scanpy as sc
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')

from src.utils import load_config, setup_logger, apply_chitin_style
from src.utils import log_memory, snapshot, force_gc
from src import engine
from src import diagnostics as diag
from src import plotting as cplt

cfg    = load_config('chitin_config.yaml')
logger = setup_logger(cfg)
apply_chitin_style(cfg)

print(f"CHITIN v2 initialized")
print(f"  Dataset       : {cfg['dataset']['name']}")
print(f"  Input dir     : {cfg['paths']['_input']}")
print(f"  Correction    : {cfg.get('correction', {}).get('mode', 'knn')}")
print(f"  Auto-calibrate: {cfg.get('calibration', {}).get('auto_calibrate', True)}")
print(f"  Fallback k    : {cfg['knn']['k']}  (overridden if auto_calibrate: true)")
print(f"  Fallback n_pcs: {cfg['knn']['n_pcs']}  (overridden if auto_calibrate: true)")
log_memory(logger, 'startup')

## 2 · Load SPORE Metacell Data

In [ ]:
train_path = cfg['paths']['_input'] / cfg['paths']['train_file']
adata_train = sc.read_h5ad(train_path)
snapshot(adata_train, 'Train metacells loaded', logger)

print(f"\n.obs columns: {list(adata_train.obs.columns)}")
print(f"Perturbation column ('{cfg['dataset']['perturbation_col']}'):")
print(adata_train.obs[cfg['dataset']['perturbation_col']].value_counts().head(10))

---
## 3 · Pre-CHITIN Diagnostics

Quantify the systematic variation **before** applying correction.  
These centroids are for evaluation only — they are never used in the transform.

In [ ]:
# Systema centroid analysis — establishes the baseline systematic variation
C_ctrl_pre, O_pert_pre, V_pre = diag.compute_systema_centroids(
    adata_train, cfg, logger)

cos_ctrl_pre, cos_pert_pre, pert_names = diag.compute_perturbation_cosines(
    adata_train, C_ctrl_pre, O_pert_pre, cfg, logger)

In [ ]:
cplt.plot_cosine_distributions(cos_ctrl_pre, cos_pert_pre, cfg, label='pre')

---
## 4 · Fit CHITIN Model (Auto-Calibration)

When `auto_calibrate: true` in the yaml, `fit()` runs a Pareto sweep over
(k, n_pcs, distance_metric) combinations and selects the optimal operating point
before fitting the final model. No manual tuning required.

Correction modes (set in yaml):
- `knn` — localized KNN subtraction (best for heterogeneous cell lines like K562, RPE1)
- `pc_decomposition` — projects out systematic PCA axes (best for homogeneous lines like iPSC, hESC)
- `hybrid` — PC decomposition then KNN on residuals (most aggressive)

In [ ]:
chitin_model = engine.ChitinModel()
adata_train  = chitin_model.fit(adata_train, cfg, logger)

if chitin_model.selected_params:
    print(f"\n  ★ Auto-selected: k={chitin_model.k}, "
          f"n_pcs={chitin_model.n_pcs}, "
          f"metric={chitin_model.distance_metric}")
else:
    print(f"\n  Manual params: k={chitin_model.k}, "
          f"n_pcs={chitin_model.n_pcs}, "
          f"metric={chitin_model.distance_metric}")

### 4a · Pareto Sweep Results

Full table of all combinations tested, and the Pareto front.  
The selected point (★) maximises rank disruption subject to signal stability constraints.

In [ ]:
from src.diagnostics import sweep_results_summary

sweep_df, pareto_df = sweep_results_summary(chitin_model, logger)

if sweep_df is not None:
    print(f"\nFull sweep ({len(sweep_df)} combinations) — top 20 by rank disruption:")
    display(sweep_df.sort_values('rank_disruption', ascending=False).head(20))
    print(f"\nPareto front ({len(pareto_df)} points):")
    display(pareto_df.sort_values('rank_disruption', ascending=False))
    cplt.plot_pareto_sweep(chitin_model, cfg)

### 4b · PC Decomposition Report

Only relevant when `correction.mode: pc_decomposition` or `hybrid`.  
Shows which PCA axes encode systematic variation and will be projected out.

In [ ]:
from src.diagnostics import pc_decomposition_report

if chitin_model.correction_mode in ('pc_decomposition', 'hybrid'):
    pc_report_df = pc_decomposition_report(
        chitin_model, adata_train, cfg, logger, top_n_genes=10)
    if pc_report_df is not None:
        print("\nPC decomposition report:")
        display(pc_report_df)
        cplt.plot_pc_systematic_cosines(chitin_model, cfg)
else:
    print(f"PC decomposition not active (mode={chitin_model.correction_mode}).")
    print("To enable: set correction.mode: 'pc_decomposition' or 'hybrid' in chitin_config.yaml")

### 4c · PCA Latent Space Visualisation

In [ ]:
cplt.plot_pca_variance(adata_train, cfg)
cplt.plot_latent_space(adata_train, cfg)

---
## 5 · Transform — Train Split

In [ ]:
adata_train_delta = chitin_model.transform(adata_train, cfg, logger, label='Train')

---
## 6 · Post-CHITIN Diagnostics

In [ ]:
# Rank-order disruption — primary metric for LightGBM effectiveness
mean_rho, rank_corrs, sampled_genes = diag.compute_rank_disruption(
    adata_train, adata_train_delta, cfg, logger)
cplt.plot_rank_disruption(rank_corrs, sampled_genes, cfg)

In [ ]:
# Pairwise perturbation discrimination
dist_pre, dist_post = diag.compute_pairwise_discrimination(
    adata_train, adata_train_delta, cfg, logger)
cplt.plot_pairwise_discrimination(dist_pre, dist_post, cfg)

In [ ]:
# Delta magnitude distribution — which perturbations have the strongest causal signal?
delta_df, delta_norms = diag.compute_delta_magnitudes(
    adata_train_delta, cfg, logger)
cplt.plot_delta_magnitude_distribution(delta_norms, cfg)
cplt.plot_top_perturbation_deltas(delta_df, cfg, n_top=30)

---
## 7 · Transform — Val & Test Splits

Uses the **training reference manifold** (fitted above) — no leakage.

In [ ]:
val_path    = cfg['paths']['_input'] / cfg['paths']['val_file']
adata_val   = sc.read_h5ad(val_path)
snapshot(adata_val, 'Val metacells loaded', logger)
adata_val_delta = chitin_model.transform(adata_val, cfg, logger, label='Val')
del adata_val
force_gc(logger)

In [ ]:
test_path   = cfg['paths']['_input'] / cfg['paths']['test_file']
adata_test  = sc.read_h5ad(test_path)
snapshot(adata_test, 'Test metacells loaded', logger)
adata_test_delta = chitin_model.transform(adata_test, cfg, logger, label='Test')
del adata_test
force_gc(logger)

---
## 8 · Save Outputs

In [ ]:
output_dir = cfg['paths']['_output']
suffix     = cfg['output']['suffix']
dataset    = cfg['dataset']['name']

splits = {'train': adata_train_delta, 'val': adata_val_delta, 'test': adata_test_delta}
for key, ad in splits.items():
    fname = f"{dataset}_{key}{suffix}.h5ad"
    path  = output_dir / fname
    ad.write_h5ad(path)
    logger.info(f"Saved {key} → {path}")
    print(f"  ✓ {key}: {ad.n_obs:,} metacells × {ad.n_vars:,} genes → {fname}")

print(f"\nAll outputs saved to: {output_dir}")
log_memory(logger, 'saved')

---
## 9 · Summary & Report

In [ ]:
print("\n" + "═" * 60)
print("  CHITIN v2 Pipeline Complete")
print("═" * 60)
print(f"  Correction mode    : {chitin_model.correction_mode}")
print(f"  k (selected)       : {chitin_model.k}")
print(f"  n_pcs (selected)   : {chitin_model.n_pcs}")
print(f"  metric (selected)  : {chitin_model.distance_metric}")
if chitin_model.systematic_pc_indices:
    print(f"  Systematic PCs out : {[f'PC{i+1}' for i in chitin_model.systematic_pc_indices]}")
print(f"  Rank disruption ρ  : {mean_rho:.4f}")
print(f"  Discrimination Δ   : {(dist_post.mean()-dist_pre.mean())/dist_pre.mean()*100:+.1f}%")
print(f"  Pre-CHITIN cos(V)  : {cos_ctrl_pre.mean():.4f}")
print(f"  Pre-CHITIN |V|     : {np.linalg.norm(V_pre):.4f}")
print(f"  Train : {adata_train_delta.n_obs:,} metacells")
print(f"  Val   : {adata_val_delta.n_obs:,} metacells")
print(f"  Test  : {adata_test_delta.n_obs:,} metacells")
print("═" * 60)
log_memory(logger, 'final')

In [ ]:
from src.reporting import generate_report

report_path = generate_report(
    cfg               = cfg,
    mean_rho          = mean_rho,
    dist_pre          = dist_pre,
    dist_post         = dist_post,
    cos_ctrl_pre      = cos_ctrl_pre,
    V_pre             = V_pre,
    delta_df          = delta_df,
    delta_norms       = delta_norms,
    adata_train_delta = adata_train_delta,
    adata_val_delta   = adata_val_delta,
    adata_test_delta  = adata_test_delta,
    chitin_model      = chitin_model,
)